# L2-02 配套：Stack 的上下文窗口、显存账与上下文敏感性配套课文：[`docs/lessons/L2-02-Stack与上下文学习.md`](../docs/lessons/L2-02-Stack与上下文学习.md)。**本课回答三件能在纯 NumPy 里验证的事：**1. 推理时的上下文窗口到底是怎么拼出来的（哪些细胞进窗口、各占多少、尾组怎么办）；2. 这个窗口的**显存与算力代价**随细胞数 `K` 怎么涨，本机 8 GiB 能跑到哪一步；3. 为什么「细胞间注意力」是上下文学习的机制来源——同一批查询细胞换一批提示细胞，输出就会变。**边界（读完再往下看）：** 这里**没有**官方 Stack 模型、没有权重、没有 torch。所有数字都是按固定 commit `cacc2e4b` 的源码形状**手算**出来的，属于工程假设。参数量那一格是个例外：手算结果与论文表格一致（Base 7,670 万 / Large 2.17 亿），可以当作一次独立核验。**耗时一个数字都没有实测**，只给 FLOPs 和一条可替换假设的换算。

In [ ]:
# -*- coding: utf-8 -*-
"""单元 0：环境。纯 NumPy + matplotlib，不需要 torch / GPU / 权重。"""
import json
import math
import os
from collections import Counter
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

SEED = 20260917
rng = np.random.default_rng(SEED)


def find_root() -> Path:
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "pyproject.toml").exists():
            return q
    return p


# 图内中文：Windows 上优先用系统 CJK 字体，找不到就退回 DejaVu（中文会变方块，但图仍可读）
_CJK = ["Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "Source Han Sans SC", "DengXian"]
_available = {f.name for f in matplotlib.font_manager.fontManager.ttflist}
for _f in _CJK:
    if _f in _available:
        plt.rcParams["font.sans-serif"] = [_f, "DejaVu Sans"]
        break
plt.rcParams["axes.unicode_minus"] = False

ROOT = find_root()
OUTDIR = ROOT / "output" / "notebook-learning" / "07-stack-icl"
os.makedirs(OUTDIR, exist_ok=True)

# Stack (Large, 后训练/对齐档) 的官方配置，来源 configs/training/bc_large.yaml、
# configs/finetuning/ft_parsecg.yaml 与论文方法节。两套 K 不同：预训练 256，后训练 512。
N_GENES = 15012       # 统一基因名单上限（论文方法节）；genelist 文件名写作 15000max
N_TOK = 100           # n_hidden，基因模块 token 数
D_TOK = 16            # token_dim
N_LAYER = 9           # n_layers
H_CELL = 8            # 细胞内注意力头数（源码硬编码，不是配置项）
H_GENE = 8            # 细胞间注意力头数（配置项 n_heads）
MLP_RATIO = 4
K_PRETRAIN = 256
K_ALIGNED = 512

print("ROOT   =", ROOT)
print("OUTDIR =", OUTDIR)
print("numpy  =", np.__version__)
print("字体    =", plt.rcParams["font.sans-serif"][0])
print("Large  : G=%d, n=%d, d=%d, nd=%d, layers=%d" % (N_GENES, N_TOK, D_TOK, N_TOK * D_TOK, N_LAYER))

## 单元 1｜上下文窗口是怎么拼出来的源码 `src/stack/models/core/inference.py::get_incontext_prediction` 做的事可以概括成三行：```textratio    = prompt_ratio + context_ratio          # 窗口里"提示侧"占多大n_test   = int(n_cells * (1 - ratio))            # 剩下的给查询（test）细胞n_base   = n_cells - n_test                      # 提示（base）细胞数```然后**逐个窗口**从 `base_adata` 取 `n_base` 个细胞、从 `test_adata` 取 `n_test` 个细胞拼成一个`n_cells` 行的 AnnData，送进模型。下面用纯 Python 复刻这个循环，重点看两个容易被忽略的行为：- **尾组不足会从头补齐**：最后一个窗口如果 `test` 细胞不够，源码用 `test_indices[:need]` 填充，  也就是**把开头的查询细胞复制一份**；- **base 用尽会循环**：`base_idx_ptr` 取模回绕，提示细胞可以被复用。对 VC2026 的直接后果：**每靶点恰好 400 个细胞**，这个数字几乎不可能被窗口整除。

In [ ]:
def build_windows(n_base_avail, n_test_avail, n_cells,
                  prompt_ratio=0.25, context_ratio=0.4):
    """复刻 get_incontext_prediction 的窗口切分逻辑（不含数据搬运）。

    返回 (windows, n_base_per, n_test_per)，windows 里每项是该窗口的
    (base 下标列表, test 下标列表)。
    """
    ratio = prompt_ratio + context_ratio
    n_test_per = max(1, int(n_cells * (1 - ratio)))
    n_base_per = n_cells - n_test_per

    n_samples = math.ceil(n_test_avail / n_test_per)
    windows, base_ptr = [], 0
    for i in range(n_samples):
        s, e = i * n_test_per, (i + 1) * n_test_per
        test_idx = list(range(s, min(e, n_test_avail)))
        if len(test_idx) < n_test_per:                 # 尾组不足 -> 从 test 开头补齐
            need = n_test_per - len(test_idx)
            test_idx += list(range(need))
        base_idx = [(base_ptr + k) % n_base_avail for k in range(n_base_per)]
        base_ptr = (base_ptr + n_base_per) % n_base_avail
        windows.append((base_idx, test_idx))
    return windows, n_base_per, n_test_per


def dedup_stats(n_test_avail, n_cells, prompt_ratio, context_ratio):
    windows, n_base_per, n_test_per = build_windows(
        10_000, n_test_avail, n_cells, prompt_ratio, context_ratio)
    flat = []
    for _, t in windows:
        flat += t
    c = Counter(flat)
    dup = sum(v - 1 for v in c.values())               # 重复出现的查询细胞数
    return n_base_per, n_test_per, len(windows), dup


print("对齐档 K=%d，查询侧 400 个细胞（本赛每靶点细胞数）" % K_ALIGNED)
print()
print("%-14s %-10s %-10s %-10s %s" % ("context_ratio", "n_base", "n_query", "窗口数", "被复制的查询细胞"))
for cr in (0.20, 0.25, 0.40):
    nb, nq, ns, dup = dedup_stats(400, K_ALIGNED, 0.25, cr)
    print("%-14.2f %-10d %-10d %-10d %d / 400" % (cr, nb, nq, ns, dup))

print()
print("预训练档 K=%d（Stack-Large 未对齐权重）" % K_PRETRAIN)
for cr in (0.25, 0.40):
    nb, nq, ns, dup = dedup_stats(400, K_PRETRAIN, 0.25, cr)
    print("  context_ratio=%.2f -> base %d / query %d / 窗口 %d / 复制 %d" % (cr, nb, nq, ns, dup))

# 提示细胞被复用的次数：当 base 池不够填满所有窗口时才会发生
windows, nb_per, nq_per = build_windows(500, 400, K_ALIGNED, 0.25, 0.4)
reuse = Counter()
for b, _ in windows:
    reuse.update(b)
print()
print("base 池只有 500 个细胞、需要 %d 个窗口时：最多被复用 %d 次" % (len(windows), max(reuse.values())))

### 单元 1 的三个结论1. **提示细胞占比不是 25%，而是 `prompt_ratio + context_ratio`。** 论文正文说的是「提示条件 25% 固定」，   但源码里决定窗口切分的是两者之和（默认 0.25 + 0.4 = 0.65）。照论文理解会把查询侧算多。2. **查询侧会被复制。** 400 个细胞配 `context_ratio=0.4` 时，有 137 个查询细胞在输出里出现两次。   这不是 bug，是源码 `test_indices[:need]` 的既定行为，但**提交前必须去重并按原顺序对齐**，   否则 400 行里会有重复行。3. **`K` 不是结构约束。** `TabularAttentionLayer.__init__` 收下 `n_cells` 但从不使用它来构造任何权重，   注意力在 `forward` 里按实际形状 reshape。所以 256 / 512 只是**推理时的窗口大小**，改它不需要重训。   这一点与 [L2-01](../docs/lessons/L2-01-State模型深潜.md) §2.4 的第 2 项完全同构。

## 单元 2｜显存账：真正的开销在"便宜"的那个轴上一层的注意力分数张量有两个：| 注意力 | 张量形状 | 元素数 | 直觉 ||---|---|---|---|| 细胞内（intra-cellular） | `(B·K, H_cell, n, n)` | `B·K·H_cell·n²` | 每个细胞一条长度 `n` 的序列，**要跑 K 条** || 细胞间（inter-cellular） | `(B, H_gene, K, K)` | `B·H_gene·K²` | 每个细胞集一条长度 `K` 的序列，**只跑 B 条** |直觉上"序列长度 100 比 512 便宜"，但细胞内注意力要对 **K 个细胞各跑一次**，于是元素数是 `K·n²` 而不是 `n²`。在 `K=512` 时它比细胞间注意力大约 **20 倍**。

In [ ]:
GIB = 1024 ** 3


def attn_scores(B, K, n=N_TOK, h_cell=H_CELL, h_gene=H_GENE):
    """一层里两个注意力分数张量的元素数。"""
    intra = B * K * h_cell * n * n      # 细胞内：B*K 条独立序列，每条 n×n
    inter = B * h_gene * K * K          # 细胞间：B 条独立序列，每条 K×K
    return intra, inter


print("%-8s %-16s %-16s %-10s" % ("K", "细胞内/层 (MB)", "细胞间/层 (MB)", "倍数"))
for K in (64, 128, 256, 512, 1024):
    a, b = attn_scores(1, K)
    print("%-8d %-16.1f %-16.1f %-10.1f"
          % (K, a * 4 / 1024 ** 2, b * 4 / 1024 ** 2, a / b))


# ---- 一次前向的峰值显存（batch=B, 窗口=K, fp32） -------------------------
PARAMS = 217_484_712        # 见单元 3 的手算结果


def peak_bytes(B, K, dtype_bytes=4, params=PARAMS, n_layer=N_LAYER):
    """粗估：权重 + 9 层注意力分数 + 输出侧缓冲。不含 allocator 碎片与 CUDA context。"""
    intra, inter = attn_scores(B, K)
    scores = (intra + inter) * n_layer * dtype_bytes
    # 输出侧：output_mlp 中间 2nd、NB 参数 2G、px_scale G、nb_mean G、nb_disp G、采样计数 G
    out = B * K * (2 * N_TOK * D_TOK + 5 * N_GENES) * dtype_bytes
    return params * dtype_bytes + scores + out


BUDGET = 8 * GIB
print()
print("一次前向峰值（fp32，含 2.17 亿参数 = %.0f MB）" % (PARAMS * 4 / 1024 ** 2))
print("%-8s %-8s %-14s %s" % ("K", "batch", "峰值 (GiB)", "结论"))
for K in (256, 512):
    fits = [B for B in range(1, 65) if peak_bytes(B, K) <= BUDGET]
    bmax = max(fits) if fits else 0
    print("%-8d %-8d %-14.2f %s" % (K, bmax, peak_bytes(bmax, K) / GIB,
                                    "8 GiB 内可行" if bmax else "batch=1 也放不下"))
    if bmax:
        for B in (bmax, bmax + 1):
            print("           batch=%-3d -> %.2f GiB" % (B, peak_bytes(B, K) / GIB))

print()
print("bf16（2 字节）同一档：")
for K in (256, 512):
    fits = [B for B in range(1, 65) if peak_bytes(B, K, 2) <= BUDGET]
    print("  K=%-4d 最大 batch = %d（%.2f GiB）" % (K, max(fits), peak_bytes(max(fits), K, 2) / GIB))

# 训练要额外背：梯度 + AdamW 两个状态量 = 权重的 3 倍
print()
print("后训练（官方 batch=8, K=512, fp32）：")
print("  权重+梯度+优化器状态 = %.2f GiB" % (PARAMS * 4 * 3 / GIB))
print("  注意力分数（含 autograd 保存 softmax）= %.2f GiB"
      % ((sum(attn_scores(8, 512)) * N_LAYER * 4 * 2) / GIB))
print("  -> 远超 8 GiB；预训练（batch=32, K=256）更不可能。")

In [ ]:
# ---- 图：显存随 K 的增长，以及 batch 边界 --------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))

Ks = np.array([64, 128, 192, 256, 384, 512, 768, 1024])
intra = np.array([attn_scores(1, int(k))[0] * 4 / 1024 ** 2 for k in Ks])
inter = np.array([attn_scores(1, int(k))[1] * 4 / 1024 ** 2 for k in Ks])
ax = axes[0]
ax.plot(Ks, intra * N_LAYER, "o-", color="#c0392b", label="细胞内 (B·K·H·n²) × 9 层")
ax.plot(Ks, inter * N_LAYER, "s-", color="#2471a3", label="细胞间 (B·H·K²) × 9 层")
ax.set_yscale("log")
ax.set_xlabel("每个细胞集的细胞数 K")
ax.set_ylabel("注意力分数显存 (MB, fp32, batch=1)")
ax.set_title("两层注意力的显存代价")
ax.grid(alpha=.3)
ax.legend(fontsize=9)

ax = axes[1]
Bs = np.arange(1, 17)
for K, col in ((256, "#2471a3"), (512, "#c0392b")):
    tot = np.array([peak_bytes(int(b), K) / GIB for b in Bs])
    ax.plot(Bs, tot, "o-", color=col, label="K=%d" % K)
ax.axhline(8, color="#555", ls="--", lw=1.2)
ax.text(1.2, 8.15, "本机 8 GiB", color="#555", fontsize=10)
ax.set_xlabel("batch（细胞集个数）")
ax.set_ylabel("一次前向峰值 (GiB, fp32)")
ax.set_title("8 GiB 下能跑到哪个 batch")
ax.grid(alpha=.3)
ax.legend(fontsize=9)

fig.suptitle("Stack (Large) 上下文窗口的显存账 —— 手算，非实测", fontsize=12)
fig.tight_layout()
for ext in ("png", "svg"):
    fig.savefig(OUTDIR / ("stack-icl-memory.%s" % ext), dpi=200)
print("saved ->", OUTDIR / "stack-icl-memory.png")

### 单元 2 的算力口径（没有实测耗时）只给 MACs/FLOPs 这一层，耗时换算依赖你自己的假设。以 `K=512, batch=1` 为例：| 部分 | 每层 MACs | 说明 ||---|---|---|| 细胞内注意力 | `K·(4nd² + 2n²d)` | `n=100, d=16` || 细胞间注意力 | `4Kd² + 2K²d` | `K=512, d=1600`，**主导项** || FFN | `K·n·8d²` | 逐 token |加上 tokenization（`G·nd`）与 `output_mlp`（`2nd² + 4ndG`，**它自己是第二大项**），一个 512 细胞窗口的前向大约 **2.4×10¹¹ FLOPs 量级**。换算成耗时需要你自己填两个数——GPU 有效算力与达成率，本机没有跑过，不代填。

## 单元 3｜参数量：一次可复核的手算`base.py::StateICLModelBase.__init__` 里的模块是确定的，按矩阵形状数一遍即可。这一格的价值在于**它可以和论文表 3 对表**：若手算与论文一致，说明我们对架构的理解没有偏差；若不一致，说明读漏了模块。

In [ ]:
def count_params(G, n=N_TOK, d=D_TOK, n_layer=N_LAYER,
                 h_gene=H_GENE, h_cell=H_CELL, mlp_ratio=MLP_RATIO):
    """按 base.py / attention.py 的模块清单手算可训练参数量。"""
    nd = n * d
    # tokenization：Linear(G, nd) + 可学习 gene 位置嵌入 (n, d)
    gene_reduction = G * nd + nd
    pos = n * d

    # 细胞内注意力：MultiHeadAttention(d) —— qkv 无偏置
    cell_qkv = d * 3 * d
    cell_proj = d * d + d
    cell_ln = 2 * d
    # 细胞间注意力：MultiHeadAttention(nd)
    gene_qkv = nd * 3 * nd
    gene_proj = nd * nd + nd
    gene_ln = 2 * nd
    # FFN：d -> mlp_ratio*d -> d
    mlp_up = d * (d * mlp_ratio) + (d * mlp_ratio)
    mlp_down = (d * mlp_ratio) * d + d
    mlp_ln = 2 * d

    per_layer = (cell_qkv + cell_proj + cell_ln
                 + gene_qkv + gene_proj + gene_ln
                 + mlp_up + mlp_down + mlp_ln)

    # 解码器：nd -> 2nd -> 2G
    out_1 = nd * (2 * nd) + (2 * nd)
    out_2 = (2 * nd) * (2 * G) + (2 * G)

    total = gene_reduction + pos + per_layer * n_layer + out_1 + out_2
    return {
        "gene_reduction": gene_reduction,
        "gene_pos": pos,
        "per_layer": per_layer,
        "layers": per_layer * n_layer,
        "output_mlp": out_1 + out_2,
        "total": total,
        "non_embedding": total - gene_reduction,
    }


for name, d, nl in (("Base", 8, 6), ("Large", 16, 9)):
    r = count_params(N_GENES, d=d, n_layer=nl)
    print("=== Stack (%s)  d=%d, layers=%d ===" % (name, d, nl))
    print("  tokenization Linear(G -> nd) : %12s" % format(r["gene_reduction"], ","))
    print("  基因位置嵌入                 : %12s" % format(r["gene_pos"], ","))
    print("  单层 TabularAttention        : %12s" % format(r["per_layer"], ","))
    print("  %d 层合计                    : %12s" % (nl, format(r["layers"], ",")))
    print("  output_mlp (nd -> 2nd -> 2G) : %12s" % format(r["output_mlp"], ","))
    print("  ---------------------------------------------")
    print("  总可训练参数                 : %12s  (%.4f 亿)"
          % (format(r["total"], ","), r["total"] / 1e8))
    print("  扣掉 tokenization 后         : %12s  (%.4f 亿)"
          % (format(r["non_embedding"], ","), r["non_embedding"] / 1e8))
    print()

print("对表：论文表 3 给的是 Base 7,670 万 / 非嵌入 6,470 万；"
      "Large 2.17 亿 / 非嵌入 1.93 亿。")
print("注意「非嵌入」只扣掉了输入侧 tokenization，1.01 亿的 output_mlp 仍算在内。")

## 单元 4｜迭代生成的进度表（T = 5）`get_incontext_generation` 的默认档是 `num_steps=5`。它排两张表：```textt      = (arange(T)+1) / T          -> [0.2, 0.4, 0.6, 0.8, 1.0]mr     = 1 - t                      -> [0.8, 0.6, 0.4, 0.2, 0.0]   # 还剩多少待预测cr     = linspace(0.2, 0.4, T)      -> [0.2, 0.25, 0.3, 0.35, 0.4] # 提示上下文占比递增```每步的窗口切分用同一个公式，所以**提示细胞数会一步步变多、查询细胞一步步变少**。

In [ ]:
def schedule(num_steps=5, prompt_ratio=0.25, context_ratio=0.4,
             context_ratio_min=0.2, n_cells=K_ALIGNED):
    t = (np.arange(num_steps, dtype=np.float32) + 1) / num_steps
    mr = 1 - t
    mr[-1] = 0.0
    cr = (np.array([context_ratio], dtype=np.float32) if num_steps == 1
          else np.linspace(context_ratio_min, context_ratio, num_steps, dtype=np.float32))
    rows = []
    for i, (m, c) in enumerate(zip(mr, cr), start=1):
        ratio = prompt_ratio + c
        n_q = max(1, int(n_cells * (1 - ratio)))
        rows.append((i, m, c, n_cells - n_q, n_q))
    return rows


print("mask 计划  :", [round(float(1 - (i + 1) / 5), 1) for i in range(5)])
print("context 计划:", [round(float(x), 3) for x in np.linspace(0.2, 0.4, 5)])
print()
print("%-6s %-12s %-14s %-12s %s" % ("步", "mask_rate", "context_ratio", "提示细胞", "查询细胞"))
for i, m, c, nb, nq in schedule():
    print("%-6d %-12.2f %-14.2f %-12d %d" % (i, m, c, nb, nq))

rows = schedule()
fig, ax = plt.subplots(figsize=(6.6, 3.8))
steps = [r[0] for r in rows]
ax.bar(steps, [r[3] for r in rows], color="#7d3c98", label="提示侧 (base)")
ax.bar(steps, [r[4] for r in rows], bottom=[r[3] for r in rows],
       color="#f39c12", label="查询侧 (test)")
ax.set_xlabel("生成步 t")
ax.set_ylabel("窗口内细胞数")
ax.set_title("T=5：提示占比逐步增加（K=512）")
ax.legend(fontsize=9)
ax.grid(alpha=.3, axis="y")
fig.tight_layout()
fig.savefig(OUTDIR / "stack-icl-schedule.png", dpi=200)
print("saved ->", OUTDIR / "stack-icl-schedule.png")

## 单元 5｜上下文敏感性：为什么"换一批提示细胞"就会换一份输出这是本课最该亲手跑的一格。构造一个 toy 的**细胞间注意力**层（就是一个标准的集合自注意力），固定住查询细胞，只换提示细胞，看查询细胞的输出变不变；然后把细胞间注意力**关掉**再跑一次。- 有细胞间注意力：换提示 → 查询输出变（这就是上下文学习）；- 关掉它：换提示 → 查询输出**一模一样**（每个细胞独立编码，上下文根本不存在）。另外顺手验证一条：**打乱提示细胞的顺序，查询输出不变**。细胞集是集合不是序列，这条性质决定了"提示细胞的排列顺序"不是一个可调的超参。

In [ ]:
def inter_cell_attention(X, Wq, Wk, Wv, Wo):
    """标准多头集合注意力的单头简化版：X (K, d) -> Y (K, d)，含残差。"""
    d = X.shape[1]
    Q, K_, V = X @ Wq, X @ Wk, X @ Wv
    scores = (Q @ K_.T) / np.sqrt(d)
    scores -= scores.max(axis=1, keepdims=True)
    A = np.exp(scores)
    A /= A.sum(axis=1, keepdims=True)
    return X + (A @ V) @ Wo


d = 32
n_query = 8
n_prompt = 16

Wq = rng.normal(0, d ** -0.5, (d, d))
Wk = rng.normal(0, d ** -0.5, (d, d))
Wv = rng.normal(0, d ** -0.5, (d, d))
Wo = rng.normal(0, d ** -0.5, (d, d))

query_cells = rng.normal(0, 1, (n_query, d))          # 固定不动
prompt_A = rng.normal(0, 1, (n_prompt, d))            # 提示集 A
prompt_B = rng.normal(3, 1, (n_prompt, d))            # 提示集 B（整体偏移）

sets = {"A": prompt_A, "B": prompt_B}
outs_with, outs_without = {}, {}
for name, P in sets.items():
    X = np.vstack([P, query_cells])                   # 源码顺序：base 在前，test 在后
    Y = inter_cell_attention(X, Wq, Wk, Wv, Wo)
    outs_with[name] = Y[n_prompt:]                    # 取查询位置的输出
    outs_without[name] = query_cells.copy()           # "关掉细胞间注意力" = 恒等映射

diff_with = np.abs(outs_with["A"] - outs_with["B"]).max()
diff_without = np.abs(outs_without["A"] - outs_without["B"]).max()
print("有细胞间注意力：换提示集后查询输出的最大逐元素差 = %.6f" % diff_with)
print("关掉细胞间注意力：同一差值                      = %.6f" % diff_without)
print()

# 置换不变性：打乱提示细胞的顺序，查询输出不应改变
perm = rng.permutation(n_prompt)
X_perm = np.vstack([prompt_A[perm], query_cells])
Y_perm = inter_cell_attention(X_perm, Wq, Wk, Wv, Wo)
print("打乱提示细胞顺序后，查询输出的差 = %.3e（应接近 0）"
      % np.abs(Y_perm[n_prompt:] - outs_with["A"]).max())

# 上下文"强度"：提示细胞数与查询输出变化量的关系
print()
print("%-14s %s" % ("提示细胞数", "换提示后查询输出的最大变化"))
for n_p in (1, 2, 4, 8, 16, 32):
    pa = rng.normal(0, 1, (n_p, d))
    pb = rng.normal(3, 1, (n_p, d))
    ya = inter_cell_attention(np.vstack([pa, query_cells]), Wq, Wk, Wv, Wo)[n_p:]
    yb = inter_cell_attention(np.vstack([pb, query_cells]), Wq, Wk, Wv, Wo)[n_p:]
    print("%-14d %.4f" % (n_p, np.abs(ya - yb).max()))

## 单元 6｜把结论带回课文跑完请把这三个数字抄进课文 §8 的检查表：1. **我的窗口切分**：`prompt_ratio` / `context_ratio` / `K` 分别是多少 → 提示侧和查询侧各多少细胞；2. **我的显存上限**：在 8 GiB 下能跑到哪个 batch；3. **我的重复行**：每靶点 400 个细胞会被复制多少行，去重方案是什么。**三件本课明确没做的事**（写进记录，不要当成已验证）：没有加载官方权重、没有跑真实推理、没有测任何耗时。参数量与论文对表一致是唯一一次和官方数字的直接碰撞。